In [15]:
import pandas as pd
from google.cloud import bigquery

In [16]:
file_path = "online_retail_II.xlsx"

df_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

df = pd.concat([df_1, df_2], ignore_index=True)

print(f"Total rows after combining: {len(df)}")
df.head()

Total rows after combining: 1067371


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [17]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [18]:
df.dropna(subset=["Customer ID", "Description"], inplace=True)

print(f"Rows after dropping nulls: {len(df)}")

Rows after dropping nulls: 824364


- Cancelled Invoices -> Prefix 'C'
- ~ -> 'NOT'

In [19]:
df = df[~df["Invoice"].astype(str).str.startswith("C")]

print(f"Rows after removing cancellations: {len(df)}")

Rows after removing cancellations: 805620


Quantity & Price > 0 ; Valid

In [20]:
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

print(f"Rows after removing invalid quantities/prices: {len(df)}")

Rows after removing invalid quantities/prices: 805549


In [21]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str)

df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID            object
Country                object
dtype: object

Revenue Column -> Revenue = Quantity * Price

In [22]:
df["Revenue"] = df["Quantity"] * df["Price"]

print(f"Revenue column sample:")
df[["Invoice", "Description", "Quantity", "Price", "Revenue"]].head()

Revenue column sample:


,Invoice,Description,Quantity,Price,Revenue
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,489434,PINK CHERRY LIGHTS,12,6.75,81.0
2,489434,WHITE CHERRY LIGHTS,12,6.75,81.0
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0


In [23]:
df.rename(columns={"Customer ID": "CustomerID"}, inplace=True)

print("Final column names:", df.columns.tolist())
print(f"Final cleaned row count: {len(df)}")

Final column names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'Revenue']
Final cleaned row count: 805549


In [24]:
df.describe()

,Quantity,InvoiceDate,Price,Revenue
count,805549.000000,805549,805549.000000,805549.000000
mean,13.290522,2011-01-02 10:24:44.106814464,3.206561,22.026505
min,1.000000,2009-12-01 07:45:00,0.001000,0.001000
25%,2.000000,2010-07-07 12:08:00,1.250000,4.950000
50%,5.000000,2010-12-03 15:10:00,1.950000,11.850000
75%,12.000000,2011-07-28 13:05:00,3.750000,19.500000
max,80995.000000,2011-12-09 12:50:00,10953.500000,168469.600000
std,143.634088,NaN,29.199173,224.041928


In [26]:
client = bigquery.Client(project="ecommerce-analysis-492810")

table_id = "ecommerce-analysis-492810.retail_data.online_retail"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()

print(f"Successfully loaded {len(df)} rows into {table_id}")

c:\Users\ulyze\.conda\envs\pyenv1\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Successfully loaded 805549 rows into ecommerce-analysis-492810.retail_data.online_retail
